# COMMOT analysis — CIVA 8-µm Visium HD

Clean, reproducible workflow for:

- re-annotating macrophage bins from `C1QC`/`CD163` expression;
- secreted CellChat communication within 50 µm;
- hypothesis-driven DLL4→NOTCH2 contact communication within 32 µm; and
- all seven CSV exports produced by the original notebook.

Run all cells from top to bottom. The input file is not modified.


In [ ]:
from pathlib import Path

import anndata as ad
import commot as ct
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from scipy.spatial.distance import cdist
from IPython.display import display


# Input and output
H5AD = Path("01_MS/CIVA_8um_LR.h5ad")
OUTDIR = Path("COMMOT_CIVA_8um_LR")
OUTDIR.mkdir(parents=True, exist_ok=True)

# AnnData fields and preprocessing
CELLTYPE_KEY = "cell_type"
GENE_KEY = "gene_name"
BIN_UM = 8.0
EXCLUDE_CELL_TYPES = {"Tissue", "Unannotated", "Unselected"}

# COMMOT analyses
SECRETED_DATABASE = "cellchat"
SECRETED_DISTANCE_UM = 50.0
MIN_CELL_PCT = 0.001
CONTACT_DATABASE = "cellchat_contact"
CONTACT_DISTANCE_UM = 32.0


In [ ]:
def collapse_gene_symbols(adata, gene_key=GENE_KEY):
    """Replace feature IDs with gene symbols and sum duplicate-symbol columns."""
    if gene_key not in adata.var.columns:
        out = adata.copy()
        out.var_names_make_unique()
        return out

    symbols = adata.var[gene_key].astype(str)
    valid = ~symbols.isin(["", "nan", "None", "NA"])
    source = adata[:, valid.to_numpy()].copy()
    symbols = symbols[valid].to_numpy()
    unique_symbols, inverse = np.unique(symbols, return_inverse=True)

    if len(unique_symbols) == len(symbols):
        source.var_names = symbols
        return source

    mapper = sparse.csr_matrix(
        (np.ones(len(inverse)), (np.arange(len(inverse)), inverse)),
        shape=(len(inverse), len(unique_symbols)),
    )
    X = source.X.tocsr() @ mapper if sparse.issparse(source.X) else np.asarray(source.X) @ mapper

    out = ad.AnnData(X=X, obs=source.obs.copy(), var=pd.DataFrame(index=unique_symbols))
    for key, value in source.obsm.items():
        out.obsm[key] = value.copy()
    return out


def looks_like_counts(X, max_values=100_000):
    values = X.data if sparse.issparse(X) else np.asarray(X).ravel()
    values = values[np.isfinite(values)]
    values = values[values != 0][:max_values]
    return len(values) > 0 and np.allclose(values, np.round(values), atol=1e-8)


def expression_values(adata, gene):
    if gene not in adata.var_names:
        raise KeyError(f"{gene!r} is not present in adata.var_names")
    values = adata[:, gene].X
    return np.asarray(values.toarray() if sparse.issparse(values) else values).ravel()


def standardize_lr_columns(df):
    """Give COMMOT's first four database columns stable names."""
    if df.shape[1] < 4:
        raise ValueError("Expected at least four ligand-receptor database columns.")
    out = df.iloc[:, :4].copy()
    out.columns = ["ligand", "receptor", "pathway", "signaling_type"]
    return out


def matrix_nnz(matrix):
    return int(matrix.count_nonzero() if sparse.issparse(matrix) else np.count_nonzero(matrix))


def score_lr_direction(adata, sender_ct, receiver_ct, df_lr, database_name):
    """Rank LR communication from one annotated cell type to another."""
    labels = adata.obs[CELLTYPE_KEY].astype(str).to_numpy()
    sender = labels == sender_ct
    receiver = labels == receiver_ct
    if not sender.any():
        raise ValueError(f"No bins found for sender type: {sender_ct}")
    if not receiver.any():
        raise ValueError(f"No bins found for receiver type: {receiver_ct}")

    possible_pairs = int(sender.sum() * receiver.sum())
    rows = []
    for ligand, receptor, pathway, *_ in df_lr.itertuples(index=False, name=None):
        ligand, receptor, pathway = str(ligand), str(receptor), str(pathway)
        key = f"commot-{database_name}-{ligand}-{receptor}"
        if key not in adata.obsp:
            continue
        directional = adata.obsp[key][sender, :][:, receiver]
        score = float(directional.sum())
        n_pairs = matrix_nnz(directional)
        rows.append({
            "sender": sender_ct,
            "receiver": receiver_ct,
            "ligand": ligand,
            "receptor": receptor,
            "pathway": pathway,
            "score": score,
            "mean_per_sender": score / sender.sum(),
            "mean_per_receiver": score / receiver.sum(),
            "n_interacting_bin_pairs": n_pairs,
            "possible_bin_pairs": possible_pairs,
            "interaction_frequency": n_pairs / possible_pairs,
        })
    columns = [
        "sender", "receiver", "ligand", "receptor", "pathway", "score",
        "mean_per_sender", "mean_per_receiver", "n_interacting_bin_pairs",
        "possible_bin_pairs", "interaction_frequency",
    ]
    return pd.DataFrame(rows, columns=columns).sort_values("score", ascending=False).reset_index(drop=True)


def incoming_lr_ranking(adata, receiver_ct, lr_sets):
    labels = adata.obs[CELLTYPE_KEY].astype(str).to_numpy()
    receiver = labels == receiver_ct
    if not receiver.any():
        raise ValueError(f"No bins found for receiver type: {receiver_ct}")

    rows = []
    for signaling_type, database_name, df_lr in lr_sets:
        for ligand, receptor, pathway, *_ in df_lr.itertuples(index=False, name=None):
            ligand, receptor, pathway = str(ligand), str(receptor), str(pathway)
            key = f"commot-{database_name}-{ligand}-{receptor}"
            if key not in adata.obsp:
                continue
            incoming = float(adata.obsp[key][:, receiver].sum())
            rows.append({
                "signaling_type": signaling_type,
                "ligand": ligand,
                "receptor": receptor,
                "pathway": pathway,
                "incoming_to_macrophages": incoming,
                "mean_per_macrophages": incoming / receiver.sum(),
            })
    columns = [
        "signaling_type", "ligand", "receptor", "pathway",
        "incoming_to_macrophages", "mean_per_macrophages",
    ]
    return pd.DataFrame(rows, columns=columns).sort_values(
        "mean_per_macrophages", ascending=False
    ).reset_index(drop=True)


def global_lr_pair_ranking(adata, lr_sets):
    rows = []
    possible_pairs = adata.n_obs ** 2
    for signaling_type, database_name, df_lr in lr_sets:
        for ligand, receptor, pathway, *_ in df_lr.itertuples(index=False, name=None):
            ligand, receptor, pathway = str(ligand), str(receptor), str(pathway)
            key = f"commot-{database_name}-{ligand}-{receptor}"
            if key not in adata.obsp:
                continue
            matrix = adata.obsp[key]
            total = float(matrix.sum())
            n_pairs = matrix_nnz(matrix)
            rows.append({
                "signaling_type": signaling_type,
                "ligand": ligand,
                "receptor": receptor,
                "pathway": pathway,
                "total_score": total,
                "mean_per_bin": total / adata.n_obs,
                "n_interacting_bin_pairs": n_pairs,
                "possible_directed_bin_pairs": possible_pairs,
                "interaction_frequency": n_pairs / possible_pairs,
            })
    columns = [
        "signaling_type", "ligand", "receptor", "pathway", "total_score",
        "mean_per_bin", "n_interacting_bin_pairs", "possible_directed_bin_pairs",
        "interaction_frequency",
    ]
    return pd.DataFrame(rows, columns=columns).sort_values(
        "total_score", ascending=False
    ).reset_index(drop=True)


In [ ]:
# Load the source object and recreate the macrophage annotation.
if not H5AD.exists():
    raise FileNotFoundError(
        f"Input not found: {H5AD.resolve()}. Update H5AD in the settings cell."
    )
adata = sc.read_h5ad(H5AD)

required_obs = {CELLTYPE_KEY, "array_col", "array_row"}
missing_obs = required_obs.difference(adata.obs.columns)
if missing_obs:
    raise KeyError(f"Missing required adata.obs columns: {sorted(missing_obs)}")

cell_type = adata.obs[CELLTYPE_KEY].astype(str).copy()
cell_type.loc[cell_type.str.contains("macrophage", case=False, na=False)] = "Unannotated"
markers = sc.get.obs_df(adata, keys=["C1QC", "CD163"])
cell_type.loc[markers["C1QC"].gt(0.1) | markers["CD163"].gt(0.1)] = "Macrophages"
adata.obs[CELLTYPE_KEY] = cell_type.astype("category")

print(adata)
display(adata.obs[CELLTYPE_KEY].value_counts().rename("n_bins").to_frame())


In [ ]:
# Use physical coordinates in µm, convert feature IDs to symbols, and remove background classes.
if "spatial" in adata.obsm:
    adata.obsm["spatial_fullres_pixel"] = adata.obsm["spatial"].copy()
adata.obsm["spatial"] = adata.obs[["array_col", "array_row"]].to_numpy(float) * BIN_UM

adata = collapse_gene_symbols(adata)
keep = ~adata.obs[CELLTYPE_KEY].astype(str).isin(EXCLUDE_CELL_TYPES)
adata = adata[keep].copy()
adata.obs[CELLTYPE_KEY] = adata.obs[CELLTYPE_KEY].cat.remove_unused_categories()

nonzero = adata.X.data if sparse.issparse(adata.X) else np.asarray(adata.X).ravel()
if len(nonzero) and np.nanmin(nonzero) < 0:
    raise ValueError("adata.X contains negative values; COMMOT requires non-negative expression.")

if looks_like_counts(adata.X):
    print("X looks like raw counts: applying normalize_total(target_sum=1e4) and log1p.")
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
else:
    print("X does not look like integer counts; assuming it is already normalized/log-transformed.")

print(f"Prepared {adata.n_obs:,} bins and {adata.n_vars:,} genes.")
print("Spatial ranges (µm):", adata.obsm["spatial"].min(axis=0), "to", adata.obsm["spatial"].max(axis=0))
display(adata.obs[CELLTYPE_KEY].value_counts().rename("n_bins").to_frame())


In [ ]:
# Secreted CellChat database: filter to expressed pairs, export, then run COMMOT.
df_lr_secreted = ct.pp.ligand_receptor_database(
    database="CellChat",
    species="human",
    signaling_type="Secreted Signaling",
)
df_lr_secreted = ct.pp.filter_lr_database(
    df_lr_secreted,
    adata,
    heteromeric=True,
    filter_criteria="min_cell_pct",
    min_cell_pct=MIN_CELL_PCT,
)
df_lr_secreted = standardize_lr_columns(df_lr_secreted)
df_lr_secreted.to_csv(OUTDIR / "CellChat_filtered_LR.csv", index=False)

print(f"Retained {len(df_lr_secreted):,} expressed secreted LR pairs.")
display(df_lr_secreted.head(20))

ct.tl.spatial_communication(
    adata,
    database_name=SECRETED_DATABASE,
    df_ligrec=df_lr_secreted,
    dis_thr=SECRETED_DISTANCE_UM,
    heteromeric=True,
    pathway_sum=True,
)
print("Secreted-signaling COMMOT analysis complete.")


In [ ]:
# Cell-contact database: isolate the database-supported DLL4→NOTCH2 pair and run COMMOT.
df_lr_contact = standardize_lr_columns(
    ct.pp.ligand_receptor_database(
        database="CellChat",
        species="human",
        signaling_type="Cell-Cell Contact",
    )
)
df_dll4_notch2 = df_lr_contact.loc[
    df_lr_contact["ligand"].eq("DLL4")
    & df_lr_contact["receptor"].str.contains("NOTCH2", na=False)
].copy()
if df_dll4_notch2.empty:
    raise ValueError("CellChat does not contain a DLL4-NOTCH2 contact interaction.")

display(df_dll4_notch2)
ct.tl.spatial_communication(
    adata,
    database_name=CONTACT_DATABASE,
    df_ligrec=df_dll4_notch2,
    dis_thr=CONTACT_DISTANCE_UM,
    heteromeric=True,
    pathway_sum=True,
)
print("DLL4→NOTCH2 contact COMMOT analysis complete.")


In [ ]:
# Optional diagnostic retained from the original notebook.
labels = adata.obs[CELLTYPE_KEY].astype(str).to_numpy()
dll4_ec = (labels == "PLVAP+ Endothelium") & (expression_values(adata, "DLL4") > 0)
notch2_mac = (labels == "Macrophages") & (expression_values(adata, "NOTCH2") > 0)
xy_ec = adata.obsm["spatial"][dll4_ec]
xy_mac = adata.obsm["spatial"][notch2_mac]

if len(xy_ec) and len(xy_mac):
    distances = cdist(xy_ec, xy_mac)
    print(f"DLL4+ PLVAP endothelial bins: {len(xy_ec)}")
    print(f"NOTCH2+ macrophage bins: {len(xy_mac)}")
    print(f"Minimum distance: {distances.min():.2f} µm")
    print(f"Median nearest EC→macrophage distance: {np.median(distances.min(axis=1)):.2f} µm")
    for threshold in (8, 16, 24, 32, 50):
        print(f"Pairs ≤{threshold} µm: {(distances <= threshold).sum():,}")
else:
    print("Distance diagnostic skipped: one or both expression-positive groups are empty.")


In [ ]:
# Build all cell-type-specific rankings and export the original CSV set.
lr_sets = [
    ("Secreted Signaling", SECRETED_DATABASE, df_lr_secreted),
    ("Cell-Cell Contact", CONTACT_DATABASE, df_dll4_notch2),
]

mac_incoming = incoming_lr_ranking(adata, "Macrophages", lr_sets)
mac_incoming.to_csv(OUTDIR / "macrophages_incoming_LR_ranking.csv", index=False)

cancer_to_ec = score_lr_direction(
    adata, "Cancer cells", "PLVAP+ Endothelium", df_lr_secreted, SECRETED_DATABASE
)
cancer_to_ec.to_csv(OUTDIR / "cancer_to_PLVAP_endothelium_LR_ranking.csv", index=False)
cancer_to_ec_vegf_angpt = cancer_to_ec.loc[
    cancer_to_ec["ligand"].str.contains("VEGF|ANGPT", case=False, na=False)
].copy()
cancer_to_ec_vegf_angpt.to_csv(
    OUTDIR / "cancer_to_PLVAP_endothelium_VEGF_ANGPT_LR.csv", index=False
)

ec_to_fib = score_lr_direction(
    adata, "PLVAP+ Endothelium", "POSTN+ Fibroblast", df_lr_secreted, SECRETED_DATABASE
)
ec_to_fib.to_csv(OUTDIR / "PLVAP_endothelium_to_POSTN_fibroblast_LR_ranking.csv", index=False)
ec_to_fib_tgfb = ec_to_fib.loc[
    ec_to_fib["ligand"].str.contains("TGFB", case=False, na=False)
].copy()
ec_to_fib_tgfb.to_csv(
    OUTDIR / "PLVAP_endothelium_to_POSTN_fibroblast_TGFB_LR.csv", index=False
)

# Correctly score the contact database (the original notebook accidentally reverted to secreted signaling here).
ec_to_mac_contact = score_lr_direction(
    adata, "PLVAP+ Endothelium", "Macrophages", df_dll4_notch2, CONTACT_DATABASE
)

display(mac_incoming.head(30))
display(cancer_to_ec_vegf_angpt)
display(ec_to_fib_tgfb)
display(ec_to_mac_contact)


In [ ]:
# Global LR ranking and final CSV export.
global_lr_ranking = global_lr_pair_ranking(adata, lr_sets)
global_lr_ranking.to_csv(OUTDIR / "global_LR_pair_ranking.csv", index=False)

exported_csvs = sorted(OUTDIR.glob("*.csv"))
print(f"Exported {len(exported_csvs)} CSV files to {OUTDIR.resolve()}:")
for path in exported_csvs:
    print(" -", path.name)

display(global_lr_ranking.head(30))
